# ensemble (lstm, xgbost)

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import talib
from typing import Tuple, Dict
import warnings

warnings.filterwarnings('ignore')

In [3]:
# lstm model definition
class LSTMModel(nn.Module):
    def __init__(self, input_size=4, hidden_size=100, num_layers=2, output_size=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

In [4]:
# feature engineering for xgboost
def create_xgb_features(df):
    df = df.copy()

    # Price-based features
    df['return_1'] = df['Close'].pct_change(1)
    df['return_3'] = df['Close'].pct_change(3)
    df['return_5'] = df['Close'].pct_change(5)
    df['return_10'] = df['Close'].pct_change(10)
    df['return_20'] = df['Close'].pct_change(20)

    # Moving averages
    df['sma_5'] = df['Close'].rolling(5).mean()
    df['sma_10'] = df['Close'].rolling(10).mean()
    df['sma_20'] = df['Close'].rolling(20).mean()

    # Distance from moving averages
    df['dist_sma5'] = (df['Close'] - df['sma_5']) / df['sma_5']
    df['dist_sma20'] = (df['Close'] - df['sma_20']) / df['sma_20']

    # Technical indicators
    df['rsi_14'] = talib.RSI(df['Close'], timeperiod=14)
    df['macd'], df['macd_signal'], df['macd_hist'] = talib.MACD(
        df['Close'], fastperiod=12, slowperiod=26, signalperiod=9
    )

    # Bollinger Bands
    df['bb_upper'], df['bb_middle'], df['bb_lower'] = talib.BBANDS(df['Close'], timeperiod=20)
    df['bb_position'] = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

    # Volatility
    df['atr_14'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    df['volatility'] = df['Close'].rolling(20).std()

    # Volume indicators
    df['volume_sma'] = df['Volume'].rolling(20).mean()
    df['volume_ratio'] = df['Volume'] / df['volume_sma']

    # Stochastic
    df['stoch_k'], df['stoch_d'] = talib.STOCH(
        df['High'], df['Low'], df['Close'],
        fastk_period=14, slowk_period=3, slowd_period=3
    )

    return df


In [5]:
# ensemble model class
class EnsembleDirectionPredictor:
    def __init__(self, sequence_length=60, threshold=0.002):
        self.sequence_length = sequence_length
        self.threshold = threshold
        self.lstm_scaler = MinMaxScaler()
        self.xgb_scaler = StandardScaler()
        self.lstm_model = None
        self.xgb_model = None
        self.xgb_features = None

    def prepare_lstm_data(self, data, create_sequences=True):
        features = ['Open', 'High', 'Low', 'Close']
        scaled_data = self.lstm_scaler.fit_transform(data[features])

        if not create_sequences:
            return scaled_data

        # create sequences
        X, y = [], []
        for i in range(self.sequence_length, len(scaled_data)):
            X.append(scaled_data[i-self.sequence_length:i])
            future_return = data['Close'].iloc[i] / data['Close'].iloc[i-1] -1
            y.append(1 if future_return > self.threshold else 0)

        return np.array(X), np.array(y)

    def prepare_xgb_data(self, data):
        # Create features
        df = create_xgb_features(data)

        # Create target
        future_return = df['Close'].pct_change(periods=1).shift(-1)
        df['target'] = np.where(future_return > self.threshold, 1, 0)

        # Drop NaN rows
        df = df.dropna()

        # Select features
        self.xgb_features = [
            'return_1', 'return_3', 'return_5', 'return_10', 'return_20',
            'dist_sma5', 'dist_sma20',
            'rsi_14', 'macd', 'macd_signal', 'macd_hist',
            'bb_position', 'atr_14', 'volatility',
            'volume_ratio', 'stoch_k', 'stoch_d'
        ]

        X = df[self.xgb_features].values
        y = df['target'].values

        return X, y, df

    def train_lstm(self, X_train, y_train, X_val, y_val, epochs=50, batch_size=32):
        """Train LSTM model"""
        print("Training LSTM model...")

        # Convert to tensors
        X_train_t = torch.FloatTensor(X_train)
        y_train_t = torch.LongTensor(y_train)
        X_val_t = torch.FloatTensor(X_val)
        y_val_t = torch.LongTensor(y_val)

        # Create datasets
        train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        # Initialize model
        self.lstm_model = LSTMModel(input_size=4, hidden_size=100, num_layers=2, output_size=2, dropout=0.3)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.lstm_model.parameters(), lr=0.001)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

        # Training loop
        best_loss = float('inf')
        patience = 10
        counter = 0

        for epoch in range(epochs):
            self.lstm_model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                output = self.lstm_model(batch_X)
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()

            # Validation
            self.lstm_model.eval()
            with torch.no_grad():
                val_output = self.lstm_model(X_val_t)
                val_loss = criterion(val_output, y_val_t)
                val_preds = torch.argmax(val_output, dim=1)
                val_acc = (val_preds == y_val_t).float().mean()

            scheduler.step(val_loss)

            # Early stopping
            if val_loss < best_loss:
                best_loss = val_loss
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

            if (epoch + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        print("LSTM training complete")
        return self.lstm_model

    def train_xgb(self, X_train, y_train, X_val, y_val):
        """Train XGBoost model"""
        print("\nTraining XGBoost model...")

        self.xgb_model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0.1,
            min_child_weight=3,
            objective='binary:logistic',
            random_state=42,
            n_jobs=-1,
            eval_metric='logloss'
        )

        self.xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=20
        )

        print("XGBoost training complete")
        return self.xgb_model

    def predict_lstm(self, X):
        """Get LSTM predictions (probabilities)"""
        self.lstm_model.eval()
        with torch.no_grad():
            X_t = torch.FloatTensor(X)
            output = self.lstm_model(X_t)
            probs = torch.softmax(output, dim=1).numpy()
        return probs[:, 1]  # Return probability of class 1 (up)

    def predict_xgb(self, X):
        """Get XGBoost predictions (probabilities)"""
        return self.xgb_model.predict_proba(X)[:, 1]

    def predict_ensemble(self, X_lstm, X_xgb, method='weighted_average', lstm_weight=0.4, xgb_weight=0.6):
        """
        Ensemble prediction combining LSTM and XGBoost

        Methods:
        - 'weighted_average': Weighted average of probabilities
        - 'voting': Hard voting (majority wins)
        - 'stacking': Use predictions as features for meta-learner (not implemented here for simplicity)
        """
        lstm_probs = self.predict_lstm(X_lstm)
        xgb_probs = self.predict_xgb(X_xgb)

        if method == 'weighted_average':
            # Weighted average
            ensemble_probs = lstm_weight * lstm_probs + xgb_weight * xgb_probs
            ensemble_preds = (ensemble_probs > 0.5).astype(int)

        elif method == 'voting':
            # Hard voting
            lstm_preds = (lstm_probs > 0.5).astype(int)
            xgb_preds = (xgb_probs > 0.5).astype(int)
            ensemble_preds = ((lstm_preds + xgb_preds) >= 1).astype(int)
            ensemble_probs = (lstm_probs + xgb_probs) / 2

        return ensemble_preds, ensemble_probs


In [6]:
# ========================================
# TRAINING PIPELINE
# ========================================
def train_ensemble_model(data_path, test_size=0.2, val_size=0.15):
    """Complete training pipeline for ensemble model"""

    # Load data
    print("Loading data...")
    data = pd.read_csv(data_path)
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.set_index('Date')
    data = data.tail(1000)

    # Initialize ensemble
    ensemble = EnsembleDirectionPredictor(sequence_length=60, threshold=0.002)

    # Prepare LSTM data
    print("\nPreparing LSTM data...")
    X_lstm, y_lstm = ensemble.prepare_lstm_data(data)

    # Prepare XGBoost data
    print("Preparing XGBoost data...")
    X_xgb, y_xgb, df_xgb = ensemble.prepare_xgb_data(data)

    # Align datasets (XGBoost has fewer samples due to feature creation)
    min_len = min(len(X_lstm), len(X_xgb))
    X_lstm = X_lstm[-min_len:]
    y_lstm = y_lstm[-min_len:]
    X_xgb = X_xgb[-min_len:]
    y_xgb = y_xgb[-min_len:]

    # Split data
    train_split = int((1 - test_size - val_size) * len(X_lstm))
    val_split = int((1 - test_size) * len(X_lstm))

    # LSTM splits
    X_lstm_train = X_lstm[:train_split]
    y_lstm_train = y_lstm[:train_split]
    X_lstm_val = X_lstm[train_split:val_split]
    y_lstm_val = y_lstm[train_split:val_split]
    X_lstm_test = X_lstm[val_split:]
    y_lstm_test = y_lstm[val_split:]

    # XGBoost splits
    X_xgb_train = X_xgb[:train_split]
    y_xgb_train = y_xgb[:train_split]
    X_xgb_val = X_xgb[train_split:val_split]
    y_xgb_val = y_xgb[train_split:val_split]
    X_xgb_test = X_xgb[val_split:]
    y_xgb_test = y_xgb[val_split:]

    print(f"\nTrain size: {len(X_lstm_train)}, Val size: {len(X_lstm_val)}, Test size: {len(X_lstm_test)}")

    # Train models
    ensemble.train_lstm(X_lstm_train, y_lstm_train, X_lstm_val, y_lstm_val)
    ensemble.train_xgb(X_xgb_train, y_xgb_train, X_xgb_val, y_xgb_val)

    # Evaluate individual models
    print("\n" + "="*60)
    print("INDIVIDUAL MODEL EVALUATION")
    print("="*60)

    lstm_preds = (ensemble.predict_lstm(X_lstm_test) > 0.5).astype(int)
    xgb_preds = (ensemble.predict_xgb(X_xgb_test) > 0.5).astype(int)

    print("\nLSTM Accuracy:", accuracy_score(y_lstm_test, lstm_preds))
    print("XGBoost Accuracy:", accuracy_score(y_xgb_test, xgb_preds))

    # Test different ensemble methods
    print("\n" + "="*60)
    print("ENSEMBLE EVALUATION")
    print("="*60)

    methods = [
        ('weighted_average', 0.4, 0.6),
        ('weighted_average', 0.5, 0.5),
        ('weighted_average', 0.3, 0.7),
        ('voting', None, None)
    ]

    best_acc = 0
    best_method = None

    for method_info in methods:
        method = method_info[0]
        if method == 'weighted_average':
            lstm_w, xgb_w = method_info[1], method_info[2]
            preds, probs = ensemble.predict_ensemble(X_lstm_test, X_xgb_test, method, lstm_w, xgb_w)
            label = f"{method} (LSTM:{lstm_w}, XGB:{xgb_w})"
        else:
            preds, probs = ensemble.predict_ensemble(X_lstm_test, X_xgb_test, method)
            label = method

        acc = accuracy_score(y_lstm_test, preds)
        print(f"\n{label}")
        print(f"Accuracy: {acc:.4f}")
        print(classification_report(y_lstm_test, preds, target_names=['Down', 'Up']))

        if acc > best_acc:
            best_acc = acc
            best_method = method_info

    print("\n" + "="*60)
    print(f"BEST METHOD: {best_method}")
    print(f"BEST ACCURACY: {best_acc:.4f}")
    print("="*60)

    return ensemble, X_lstm_test, X_xgb_test, y_lstm_test

In [7]:
# ========================================
# USAGE EXAMPLE
# ========================================
if __name__ == "__main__":
    ensemble, X_lstm_test, X_xgb_test, y_test = train_ensemble_model(
        '../data/aapl.us.csv',
        test_size=0.2,
        val_size=0.15
    )

    # Make predictions with best method
    final_preds, final_probs = ensemble.predict_ensemble(
        X_lstm_test, X_xgb_test,
        method='weighted_average',
        lstm_weight=0.4,
        xgb_weight=0.6
    )

    print("\nFinal Ensemble Performance:")
    print(f"Accuracy: {accuracy_score(y_test, final_preds):.4f}")

Loading data...

Preparing LSTM data...
Preparing XGBoost data...

Train size: 611, Val size: 141, Test size: 188
Training LSTM model...
Epoch [10/50], Val Loss: 0.6876, Val Acc: 0.5532
Epoch [20/50], Val Loss: 0.6874, Val Acc: 0.5532
Epoch [30/50], Val Loss: 0.6877, Val Acc: 0.5532
Early stopping at epoch 31
LSTM training complete

Training XGBoost model...
[0]	validation_0-logloss:0.68977
[20]	validation_0-logloss:0.68745
[40]	validation_0-logloss:0.69931
[60]	validation_0-logloss:0.70625
[80]	validation_0-logloss:0.71408
[100]	validation_0-logloss:0.72153
[120]	validation_0-logloss:0.72825
[140]	validation_0-logloss:0.73420
[160]	validation_0-logloss:0.73806
[180]	validation_0-logloss:0.74375
[199]	validation_0-logloss:0.73859
XGBoost training complete

INDIVIDUAL MODEL EVALUATION

LSTM Accuracy: 0.5372340425531915
XGBoost Accuracy: 0.5478723404255319

ENSEMBLE EVALUATION

weighted_average (LSTM:0.4, XGB:0.6)
Accuracy: 0.4840
              precision    recall  f1-score   support

  